## THE CORRECT packages :/

In [1]:
!pip install --upgrade pip
!pip install open3d
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
!pip install "numpy<2.0.0"

Looking in indexes: https://download.pytorch.org/whl/cu121


In [2]:
# !python -c "import open3d.ml.torch as ml3d; print('OK')"
# !python -c "import torch; print(torch.cuda.is_available())"

## Environment Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
import os

GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/THESIS"
print(os.listdir(GOOGLE_DRIVE_PATH))

['pointnet-master', 'pointnet', 'open3d-ml', 'code', 'PointPainting', 'dataset', '.git', 'README.md']


## Import pretrained model to segment own pcd file
avoid loading semantickitti dataset

In [19]:
from posix import pipe
import open3d.ml as _ml3d
import open3d.ml.torch as ml3d
import torch

framework = "torch"
device = "gpu" if torch.cuda.is_available() else "cpu"

# fetch config
# randlanet_semantickitti
cfg_file = os.path.join(GOOGLE_DRIVE_PATH, "open3d-ml/Open3D-ML/ml3d/configs/randlanet_semantickitti.yml")
cfg = _ml3d.utils.Config.load_from_file(cfg_file)

# model and pipeline
Model = _ml3d.utils.get_module("model", cfg.model.name, framework)
model = Model(**cfg.model)
pipeline = ml3d.pipelines.SemanticSegmentation(
    model,
    device=device,
    **cfg.pipeline
)

# weights
ckpt_folder = os.path.join(GOOGLE_DRIVE_PATH, "code", "logs")
os.makedirs(ckpt_folder, exist_ok=True)
ckpt_path = os.path.join(ckpt_folder, "randlanet_semantickitti_202201071330utc.pth")
randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"
if not os.path.exists(ckpt_path):
    cmd = "wget {} -O {}".format(randlanet_url, ckpt_path)
    os.system(cmd)
pipeline.load_ckpt(ckpt_path=ckpt_path)

In [27]:
import open3d as o3d
import numpy as np

pcd_path = os.path.join(GOOGLE_DRIVE_PATH, "code/016653.pcd")
pcd = o3d.io.read_point_cloud(pcd_path)
points = np.asarray(pcd.points).astype(np.float32)


xyz = np.asarray(pcd.points)
# torch.tensor(np.float32(points))
# Set the points to the correct format for inference
data = {"point":torch.tensor(xyz),
        'feat': None,
        'label':torch.tensor(np.zeros((len(xyz),), dtype=np.int32))}


result = pipeline.run_inference(data)

print(result["predict_labels"])


test 0/1: 100%|██████████| 27586/27586 [26:54<00:00, 17.08it/s]


[ 4  4  4 ... 10  8  8]


/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/pipelines/semantic_segmentation.py:179: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(data['label']), model.cfg.num_classes,
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))


In [21]:
np.save("/content/drive/My Drive/THESIS/predict_labels.npy", result["predict_labels"])

In [ ]:
# print(result["predict_labels"].shape)
# print(xyz.shape)
# # o3d.io.write_point_cloud("/content/drive/My Drive/THESIS/segmentation_result.ply", pcd)

(28008,)
(28008, 3)


True

In [87]:
COLOR_MAP = {
    -1: (192, 192, 192), # unlabeled
    0: (102, 204, 0),   # car
    1: (245, 150, 100),
    2: (245, 230, 100),
    3: (150, 60, 30),
    4: (180, 30, 80),   # back
    5: (255, 0., 0),
    6: (30, 30, 255),
    7: (200, 40, 255),
    8: (90, 30, 150),   # close ground
    9: (255, 0, 255),   # front right ground
    10: (255, 150, 255),  # front right ground
    11: (75, 0, 75),    # far ground
    12: (75, 0., 175),  # building
    13: (0, 200, 255),  # car and curbs
    14: (0, 175, 0),    # tree
    15: (50, 20, 150),
    16: (0, 60, 135),   # curbs
    17: (0, 128, 255),  # building
    18: (150, 240, 255),
    19: (0, 0, 255),
}

COLOR_MAP = {k: tuple(np.array(v)/255.0) for k, v in COLOR_MAP.items()}

# # ------ for custom data -------
# kitti_labels = {
#     0: 'unlabeled',
#     1: 'car',
#     2: 'bicycle',
#     3: 'motorcycle',
#     4: 'truck',
#     5: 'other-vehicle',
#     6: 'person',
#     7: 'bicyclist',
#     8: 'motorcyclist',
#     9: 'road',
#     10: 'parking',
#     11: 'sidewalk',
#     12: 'other-ground',
#     13: 'building',
#     14: 'fence',
#     15: 'vegetation',
#     16: 'trunk',
#     17: 'terrain',
#     18: 'pole',
#     19: 'traffic-sign'
# }

In [10]:
points = xyz
labels = result["predict_labels"]

num_classes = labels.max() + 1
colors = np.random.rand(num_classes, 3)
point_colors = colors[labels]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(point_colors)
o3d.io.write_point_cloud("/content/drive/My Drive/THESIS/segmentation_result1.ply", pcd)

True

In [89]:
labels_cars = -1 * np.ones(len(labels), dtype=int)
labels_cars[(labels == 0)] = 0  # car
labels_cars[(labels == 17)] = 17  # building
labels_cars[(labels == 12)] = 17  # building
labels_cars[(labels == 16)] = 16  # curbs

unique_labels_car, counts = np.unique(labels_cars, return_counts=True)

print("Labels:", unique_labels_car)
print("Counts:", counts)

point_colors = np.array([COLOR_MAP[l] for l in labels_cars])

pcd_car = o3d.geometry.PointCloud()
pcd_car.points = o3d.utility.Vector3dVector(points)
pcd_car.colors = o3d.utility.Vector3dVector(point_colors)
o3d.io.write_point_cloud("/content/drive/My Drive/THESIS/segmentation_result_picks.ply", pcd_car)

Labels: [-1  0 16 17]
Counts: [22022  1430  2036  2520]


True

In [67]:
print(labels.shape, labels.dtype)
print(labels_cars.shape, labels_cars.dtype)

(28008,) int64
(28008,) int64
